<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-21-April-14-2026/Lecture-21_DimensionalityReduction-Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 21 - Dimensionality Reduction - Autoencoder


Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Download datasets

%%bash
datasets="
colvar.0.data
colvar.1.data
colvar.2.data
colvar.3.data
colvar.HeavyAtoms-Distances.0.data
colvar.HeavyAtoms-Distances.1.data
colvar.HeavyAtoms-Distances.2.data
colvar.HeavyAtoms-Distances.3.data
Temperatures.txt
"

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Lecture-20-April-9-2026/Datasets"

for d in ${datasets}
do
  wget ${url}/${d} &> /dev/null
done

ls

In [ ]:
!cat Temperatures.txt

Load data

In [ ]:
def get_variables_names_from_header(filename):
  with open(filename, 'r') as f:
    header = f.readline()
    variables = header.split()[2:]
  return variables

In [ ]:
data_dih = {}

dih_variables = get_variables_names_from_header("colvar.0.data")

data_dih['300K'] = pd.read_csv("colvar.0.data", header=None, names=dih_variables, sep='\\s+', comment="#")
data_dih['416K'] = pd.read_csv("colvar.1.data", header=None, names=dih_variables, sep='\\s+', comment="#")
data_dih['576K'] = pd.read_csv("colvar.2.data", header=None, names=dih_variables, sep='\\s+', comment="#")
data_dih['800K'] = pd.read_csv("colvar.3.data", header=None, names=dih_variables, sep='\\s+', comment="#")

In [ ]:
dist_variables = get_variables_names_from_header("colvar.HeavyAtoms-Distances.0.data")
data_dist = {}

data_dist['300K'] = pd.read_csv("colvar.HeavyAtoms-Distances.0.data", header=None, names=dist_variables, sep='\\s+', comment="#")
data_dist['416K'] = pd.read_csv("colvar.HeavyAtoms-Distances.1.data", header=None, names=dist_variables, sep='\\s+', comment="#")
data_dist['576K'] = pd.read_csv("colvar.HeavyAtoms-Distances.2.data", header=None, names=dist_variables, sep='\\s+', comment="#")
data_dist['800K'] = pd.read_csv("colvar.HeavyAtoms-Distances.3.data", header=None, names=dist_variables, sep='\\s+', comment="#")

First we consider the dihedral angles

In [ ]:
show_kdeplot = False

for T in data_dih.keys():
  print(T)
  x_label = 'phi'
  y_label = 'psi'

  t = data_dih[T]['time']
  x = data_dih[T][x_label]
  y = data_dih[T][y_label]


  plt.plot(t, x, '.', label='x')
  plt.xlabel('time')
  plt.ylabel(x_label)
  plt.show()

  plt.plot(t, y, '.', label='y')
  plt.xlabel('time')
  plt.ylabel(y_label)
  plt.show()

  g = sns.JointGrid(height=7, ratio=5)
  ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
  sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0)
  ax_scatter.set_xlabel(x_label)
  ax_scatter.set_ylabel(y_label)
  sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
  sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
  plt.show()

  if show_kdeplot:
    g = sns.JointGrid(height=7, ratio=5)
    ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
    sns.kdeplot(x=x, y=y,
                ax=ax_joint,
                bw_adjust=0.6,
                fill=True,
                levels=8)
    ax_joint.set_xlabel(x_label)
    ax_joint.set_ylabel(y_label)
    sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
    sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
    plt.show()



In [ ]:
g = sns.JointGrid(height=7, ratio=5)
ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y

for T in ['800K', '576K', '416K', '300K']:
  print(T)
  x_label = 'phi'
  y_label = 'psi'

  t = data_dih[T]['time']
  x = data_dih[T][x_label]
  y = data_dih[T][y_label]




  sc = ax_scatter.scatter(x, y, s=14, alpha=0.4, linewidths=0,label=T)
  ax_scatter.set_xlabel(x_label)
  ax_scatter.set_ylabel(y_label)
  sns.kdeplot(x=x, ax=ax_top, fill=False, bw_adjust=0.6,label=T)
  sns.kdeplot(y=y, ax=ax_right, fill=False, bw_adjust=0.6)
ax_scatter.legend()
plt.show()


In [ ]:
for T in data_dih.keys():
  print(T)
  phi = data_dih[T]['phi'].to_numpy()

  data_dih[T]['C7eq_basin'] = np.where( (phi > 0) & (phi < 2) ,0 , 1)
  data_dih[T]['C7eq_bool'] = np.where( (phi > 0) & (phi < 2) ,False , True)
  data_dih[T]['C7ax_bool'] = np.where( (phi > 0) & (phi < 2) ,True , False)

  import matplotlib.colors as mcolors
  cmap = mcolors.ListedColormap(['red', 'blue'])
  plt.scatter(data_dih[T]['phi'], data_dih[T]['psi'],c=data_dih[T]['C7eq_basin'],s=10, cmap=cmap)
  cbar = plt.colorbar(ticks=[0.25,0.75])
  cbar.ax.set_yticklabels(['C7ax', 'C7_eq'])
  plt.show()

In [ ]:
import torch
# Check if CUDA is available
print(torch.cuda.is_available())  # True = GPU available

# See which device you're on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # 'cuda' or 'cpu'

# Get GPU name (if available)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # e.g. 'NVIDIA GeForce RTX 3090'
    print(torch.cuda.device_count())      # Number of GPUs

## Filter Distances



In [ ]:
T = '300K'
StdDev = []
for k in data_dist[T].keys()[1:]:
  StdDev.append(data_dist[T][k].std())


In [ ]:
StdDev_Threshold=0.01
plt.plot(np.sort(StdDev),'o-')
plt.axhline(y=StdDev_Threshold, color='r', linestyle='--', linewidth=2)
plt.show()

In [ ]:
filtered_distances = []

StdDev = []
for k in data_dist[T].keys()[1:]:
  std = data_dist[T][k].std()
  if std > StdDev_Threshold:
    filtered_distances.append(k)
print(len(filtered_distances))
print(filtered_distances)

In [ ]:
for k in data_dist[T].keys()[1:]:
  if k in filtered_distances:
    color = "blue"
    title = f"{k:s} - Included"
  else:
    color = "red"
    title = f"{k:s} - Excluded"
  sns.kdeplot(data_dist[T][k],color=color)
  plt.ylabel("Density")
  plt.xlabel(k)
  plt.title(title)
  plt.show()

## Autoencoder Code

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Dataset

class DistanceDataset(Dataset):
    """
    Expects frames: np.ndarray of shape (n_frames, n_distances)
    where n_distances = N*(N-1)//2 (upper triangle of pairwise distance matrix).
    """
    def __init__(self, frames: np.ndarray, normalize: bool = True):
        self.data = torch.from_numpy(frames).float()
        self.mean = self.data.mean(0) if normalize else 0.0
        self.std  = self.data.std(0).clamp(min=1e-8) if normalize else 1.0
        self.data = (self.data - self.mean) / self.std

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# Model

class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 2,
                 hidden_dims: list[int] = None, dropout_probablity=0.1):
        super().__init__()
        if hidden_dims is None:
            # Gradually compress: input → 256 → 128 → latent
            d = max(input_dim // 4, latent_dim * 4)
            hidden_dims = [min(input_dim, 512), d]

        # ── Encoder ──
        enc_layers = []
        in_dim = input_dim
        for h in hidden_dims:
            enc_layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_probablity),
            ]
            in_dim = h
        enc_layers.append(nn.Linear(in_dim, latent_dim))
        self.encoder = nn.Sequential(*enc_layers)

        # ── Decoder ──
        dec_layers = []
        in_dim = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_probablity),
            ]
            in_dim = h
        # Softplus keeps output ≥ 0 (distances are non-negative)
        dec_layers += [nn.Linear(in_dim, input_dim), nn.Softplus()]
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        return self.decode(z), z


# Training

def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    criterion = nn.MSELoss()
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, _ = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    criterion = nn.MSELoss()
    for batch in loader:
        batch = batch.to(device)
        recon, _ = model(batch)
        total_loss += criterion(recon, batch).item() * len(batch)
    return total_loss / len(loader.dataset)


# disable gradient calculation for complete function
@torch.no_grad()
def get_latent(model, loader, device) -> np.ndarray:
    """Return all latent vectors as a numpy array (n_frames, latent_dim)."""
    model.eval()
    zs = [model.encode(batch.to(device)).cpu() for batch in loader]
    return torch.cat(zs).numpy()

In [ ]:
T='300K'
stride=1
use_filtered_distances = True

if use_filtered_distances:
  distances = data_dist[T][filtered_distances].to_numpy()[::stride]
else:
  distances = data_dist[T].drop(columns=['time']).to_numpy()[::stride]

N_DISTANCES    = distances.shape[1]
N_FRAMES   = distances.shape[0]
LATENT_DIM = 2

print("Number of distances:",N_DISTANCES)
print("Number of frames:",N_FRAMES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Loading Dataset
dataset = DistanceDataset(distances, normalize=True)
n_val   = int(0.2 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [len(dataset) - n_val, n_val]
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

NN_multiplier = 0.75
NN_NumLayers = 3

# ── Model ──
model = Autoencoder(
    input_dim   = N_DISTANCES,
    latent_dim  = LATENT_DIM,
    hidden_dims = NN_NumLayers*[int(NN_multiplier*N_DISTANCES)],
    # hidden_dims = [256, 128, 64, 32],
    dropout_probablity=0.1
).to(device)
print(model)



In [ ]:
BATCH_SIZE = 1000
LR         = 1e-3
EPOCHS     = 200


loss_curve = []

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

# Training loop
best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train(model, train_loader, optimizer, device)
    val_loss   = evaluate(model, val_loader, device)
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    loss_curve.append([epoch,train_loss,val_loss,current_lr])

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  lr={current_lr:.4e}")

    if val_loss < best_val:
        best_val = val_loss
        # torch.save(model.state_dict(), "best_autoencoder.pt")

print(f"\nBest validation MSE: {best_val:.4f}")

loss_curve = np.array(loss_curve)
loss_curve = pd.DataFrame(loss_curve, columns=['epoch','train_loss','val_loss','lr'])

# Extract latent space for downstream analysis
all_loader = DataLoader(dataset, batch_size=BATCH_SIZE)
z = get_latent(model, all_loader, device)   # shape: (N_FRAMES, LATENT_DIM)
# np.save("latent_coords.npy", z)
# print(f"Latent representation saved: {z.shape}")

In [ ]:
plt.plot(loss_curve['epoch'],loss_curve['train_loss'],".-",label="Training Loss")
plt.plot(loss_curve['epoch'],loss_curve['val_loss'],".-",label="Validation Loss")
plt.legend()
plt.xlabel('epoch')
plt.ylabel('Loss')
plt.show()

plt.plot(loss_curve['epoch'],loss_curve['lr'],".-",label="Training Loss")
plt.legend()
plt.xlabel('epoch')
plt.ylabel('learning rate')
plt.show()


In [ ]:
plt.scatter(z[:, 0], z[:, 1], c=data_dih[T]['C7eq_basin'].iloc[::stride], s=5, alpha=0.5)
plt.show()

## Variational Autoencoder

In [ ]:
# Model

class VariationalAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=8, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            d = max(input_dim // 4, latent_dim * 4)
            hidden_dims = [min(input_dim, 512), d]

        # ── Encoder (outputs features, not z directly) ──
        enc_layers = []
        in_dim = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(in_dim, h),
                           nn.BatchNorm1d(h),
                           nn.ReLU(inplace=True)]
            in_dim = h
        self.encoder = nn.Sequential(*enc_layers)

        # Two separate heads for μ and log σ²
        self.fc_mu      = nn.Linear(in_dim, latent_dim)
        self.fc_log_var = nn.Linear(in_dim, latent_dim)

        # ── Decoder (unchanged) ──
        dec_layers = []
        in_dim = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(in_dim, h),
                           nn.BatchNorm1d(h),
                           nn.ReLU(inplace=True)]
            in_dim = h
        dec_layers += [nn.Linear(in_dim, input_dim), nn.Softplus()]
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)   # random noise ~ N(0, I)
            return mu + eps * std         # z ~ N(μ, σ²)
        return mu                         # at inference, just use the mean

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var

def vae_loss(recon, x, mu, log_var, beta=1.0):
    # Reconstruction: how well did we rebuild the distances?
    recon_loss = nn.functional.mse_loss(recon, x, reduction="sum")

    # KL divergence: how close is q(z|x) = N(μ,σ²) to p(z) = N(0,I)?
    # Closed-form solution: -0.5 * Σ(1 + log σ² - μ² - σ²)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

    return (recon_loss + beta * kl_loss) / x.size(0)  # normalize by batch size

# Training

def train_vae(model, loader, optimizer, device, beta=1.0):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, log_var = model(batch)
        loss = vae_loss(recon, batch, mu, log_var, beta=beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(batch)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate_vae(model, loader, device,beta=1.0):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        recon, mu, log_var = model(batch)
        loss = vae_loss(recon, batch, mu, log_var, beta=beta)
        total_loss += loss.item() * len(batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def get_latent_vae_mean(model, loader, device):
    """
    Returns μ only — the deterministic latent representation.
    Best for clustering, RMSD regression, free energy landscapes.
    """
    model.eval()
    mus = [model.encode(batch.to(device))[0].cpu() for batch in loader]
    return torch.cat(mus).numpy()  # shape: (n_frames, latent_dim)

In [ ]:
T='300K'
stride=1
use_filtered_distances = True

if use_filtered_distances:
  distances = data_dist[T][filtered_distances].to_numpy()[::stride]
else:
  distances = data_dist[T].drop(columns=['time']).to_numpy()[::stride]


N_DISTANCES    = distances.shape[1]
N_FRAMES   = distances.shape[0]
LATENT_DIM = 2

print("Number of distances:",N_DISTANCES)
print("Number of frames:",N_FRAMES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Loading Dataset
dataset = DistanceDataset(distances, normalize=True)
n_val   = int(0.2 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [len(dataset) - n_val, n_val]
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

NN_multiplier = 0.75
NN_NumLayers = 3

# ── Model ──
model = VariationalAutoencoder(
    input_dim   = N_DISTANCES,
    latent_dim  = LATENT_DIM,
    hidden_dims = NN_NumLayers*[int(NN_multiplier*N_DISTANCES)],
    # hidden_dims = [256, 128, 64, 32]
).to(device)
print(model)



In [ ]:
BATCH_SIZE = 1000
LR         = 1e-3
EPOCHS     = 200
BETA=1.0

loss_curve = []

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

# Training loop
best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_vae(model, train_loader, optimizer, device, BETA)
    val_loss   = evaluate_vae(model, val_loader, device, BETA)
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    loss_curve.append([epoch,train_loss,val_loss,current_lr])

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  lr={current_lr:.4e}")

    if val_loss < best_val:
        best_val = val_loss
        # torch.save(model.state_dict(), "best_autoencoder.pt")

print(f"\nBest validation Loss: {best_val:.4f}")

loss_curve = np.array(loss_curve)
loss_curve = pd.DataFrame(loss_curve, columns=['epoch','train_loss','val_loss','lr'])

# Extract latent space for downstream analysis
all_loader = DataLoader(dataset, batch_size=BATCH_SIZE)
z = get_latent_vae_mean(model, all_loader, device)   # shape: (N_FRAMES, LATENT_DIM)
# np.save("latent_coords.npy", z)
# print(f"Latent representation saved: {z.shape}")

In [ ]:
plt.plot(loss_curve['epoch'],loss_curve['train_loss'],".-",label="Training Loss")
plt.plot(loss_curve['epoch'],loss_curve['val_loss'],".-",label="Validation Loss")
plt.legend()
plt.xlabel('epoch')
plt.ylabel('Loss')
plt.show()

plt.plot(loss_curve['epoch'],loss_curve['lr'],".-",label="Training Loss")
plt.legend()
plt.xlabel('epoch')
plt.ylabel('learning rate')
plt.show()

In [ ]:
plt.scatter(z[:, 0], z[:, 1], c=data_dih[T]['C7eq_basin'].iloc[::stride], s=5, alpha=0.5)
plt.show()